# Polynomial surface and the adequacy criteria

Reproduces the polynomial-model study of Zelensky & Donchenko (2025) on the
synthetic pit: fit least-squares polynomial surfaces of increasing degree and
judge each one by the paper's three indicators, the coefficient of
determination R2, the variance ratio between data and surface, and the lag-1
autocorrelation of the input sequence.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasciencedynamics/pitmap/blob/main/notebooks/02_polynomial_surface.ipynb)

In [ ]:
# On Colab (or any fresh environment) install pitmap from GitHub first.
# Skip this cell if you already ran `make requirements` locally.
%pip install -q git+https://github.com/datasciencedynamics/pitmap.git matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pitmap import PolynomialSurface, select_degree
from pitmap.synthetic import grid, sample_survey_points, true_surface

xy, z = sample_survey_points(n=400, seed=0)
ax_x, ax_y, nodes = grid(step=30.0)
z_true = true_surface(nodes[:, 0], nodes[:, 1])

## 1. Degree sweep

One fit per degree. RMSE is against the true surface at the grid nodes; the
other three columns are the adequacy indicators computed from the survey
points alone, which is all a surveyor would have.

The lag-1 autocorrelation is a property of the input sequence, not of the fit. Here the survey points are in random order, so it is near zero; along a borehole or a survey line it would be high.

In [ ]:
fits = {d: PolynomialSurface(degree=d).fit(xy, z) for d in range(6)}

print(f"{'degree':>6} {'R2':>6} {'var ratio %':>12} {'adequate':>9} {'RMSE m':>7}")
for d, m in fits.items():
    err = np.sqrt(np.mean((m.predict(nodes) - z_true) ** 2))
    print(f"{d:>6} {m.r2_:>6.3f} {m.variance_ratio_:>12.1f} {str(m.is_adequate()):>9} {err:>7.2f}")

print("lag-1 autocorrelation of the input sequence:", round(fits[2].lag1_autocorrelation_, 3))
print("lowest adequate degree:", select_degree(xy, z).degree)

## 2. Surfaces

Low degrees cannot bend enough to follow the pit; the adequacy checks reject
them on the variance ratio, since the surface is far flatter than the data.

In [ ]:
show = [("True surface", z_true), ("Degree 2", fits[2].predict(nodes)), ("Degree 4", fits[4].predict(nodes))]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
shape = (ax_y.size, ax_x.size)
for ax, (name, values) in zip(axes, show):
    im = ax.imshow(values.reshape(shape), extent=[ax_x[0], ax_x[-1], ax_y[0], ax_y[-1]],
                   origin="lower", cmap="terrain", vmin=z_true.min(), vmax=z_true.max())
    ax.set_title(name)
    ax.set_aspect("equal")
fig.colorbar(im, ax=axes, label="Elevation, m")
plt.show()

## References

Zelensky, O., & Donchenko, O. (2025). Mining Journal of Kryvyi Rih National
University, 59(1), 10-21. https://doi.org/10.31721/2306-5435-2025-1-10-21